# Phase 4: Multi-Task Learning Training Pipeline (FIXED - Entity-Level Validation)

**Key Improvements**:
- ✅ CRF layer for valid tag transition learning
- ✅ Entity-level F1 validation (not token-level)
- ✅ Class weights to boost I-tag predictions
- ✅ B/I ratio monitoring to detect fragmentation

**Expected Improvement**: 16.77% → 60-70% entity-level F1

---

**Purpose**: Joint training of classification + NER with metadata integration and proper entity-level validation

**Author**: GBC Biodata Inventory Team  
**Created**: 2025-11-06  
**Environment**: Google Colab with GPU (optimized for A100)

## Overview

This notebook implements Phase 4 multi-task learning with critical fixes:
- **Shared RoBERTa encoder** for both classification and NER tasks
- **Metadata integration** (28 features) via projection and fusion layers
- **CRF layer** for learning valid BIO tag transitions
- **Entity-level F1 validation** (not token-level)
- **Class weights** to prevent multi-word entity fragmentation
- **Fixed loss weighting**: λ₁=0.3 (classification), λ₂=0.7 (NER), λ₃=0.1 (auxiliary)
- **A100 optimizations**: Mixed precision training, batch size 32

## Problem Fixed

**Original Issue**: Token-level loss and validation caused entity fragmentation
- Example: "Mouse Phenome Database" → ["Mouse", "Phenome", "Database"] (3 wrong entities)
- Root cause: Optimizing for token accuracy instead of entity correctness

**Solution**: CRF layer + entity-level F1 validation
- CRF learns: B-COM → I-COM is valid, B-COM → B-COM is penalized
- Validation counts: "Mouse Phenome Database" = 1 entity (not 3 tokens)

## Phase 4 Goals

- **Primary**: Entity-level NER F1 ≥ 60% (up from 16.77%)
- **Secondary**: Maintain classification F1 ~ 0.898 (V2 baseline)
- **Hypothesis**: Multi-task learning + metadata + CRF improves entity recognition

## Features

- **TEST_MODE**: Quick validation with 50 samples and 3 epochs
- **Session Tracking**: Unique session IDs for reproducibility
- **Checkpointing**: Save best classification, best NER, best combined models
- **B/I Ratio Monitoring**: Detect entity fragmentation issues
- **Google Drive Integration**: Persistent storage and archival

## Session Management

Each training session gets a unique ID: `YYYY-MM-DD-abcdef`
All outputs are saved to:
- Local: `experiments/{session_id}/`
- Drive: `MyDrive/inventory_2022/experiment_archives/{session_id}/`

---

## 🖥️ Colab Setup Requirements

**Before running this notebook:**

1. **GPU Runtime**: 
   - Go to: Runtime → Change runtime type → Hardware accelerator → **GPU**
   - **Recommended**: A100 GPU for fastest training (~1.5 hours)
   - **Alternative**: T4 or V100 GPU (~3-5 hours)

2. **RAM Setting**:
   - Select **High-RAM** if available (25.5 GB recommended)
   - Standard RAM (12.7 GB) should work but monitor usage

3. **Session Duration**:
   - **TEST_MODE=True**: 5-10 minutes
   - **TEST_MODE=False**: 1.5-5 hours (depending on GPU)
   - Keep tab open or enable browser notifications

4. **Important Notes**:
   - Colab may disconnect after 12 hours of inactivity
   - Training includes automatic checkpointing
   - All outputs saved to Google Drive for persistence

⚠️ **First-time users**: Run with `TEST_MODE = True` first to verify setup (takes ~5 min)

## Cell 1: Mount Google Drive and Setup Session

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
import datetime
import random
import string

# Mount drive
drive.mount('/content/drive', force_remount=True)

# Set base paths
PROJECT_NAME = "inventory_2022"
DRIVE_BASE = f"/content/drive/MyDrive/{PROJECT_NAME}"

# Change to project directory
os.chdir(DRIVE_BASE)

print(f"✅ Mounted Google Drive")
print(f"📁 Working directory: {os.getcwd()}")

# Generate unique session ID: YYYY-MM-DD-abcdef
date_str = datetime.datetime.now().strftime("%Y-%m-%d")
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
SESSION_ID = f"{date_str}-{random_suffix}"

print(f"\n🔬 Session ID: {SESSION_ID}")

# Create experiment directories
EXPERIMENT_DIR = f"experiments/{SESSION_ID}"
ARCHIVE_DIR = f"{DRIVE_BASE}/experiment_archives/{SESSION_ID}"

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

print(f"📂 Experiment directory: {EXPERIMENT_DIR}")
print(f"📦 Archive directory: {ARCHIVE_DIR}")

## Cell 2: Configuration - Phase 4 Multi-Task Learning with Entity-Level Validation

In [ ]:
# ========================================
# PHASE 4 CONFIGURATION - MULTI-TASK LEARNING (FIXED)
# ========================================

# TEST MODE: Set to True for quick validation (50 samples, 3 epochs)
TEST_MODE = False  # Set to True for testing, False for full training

# ========================================
# TRAINING CONFIGURATION
# ========================================
# Based on successful TEST_MODE validation run
# Fixed hyperparameters for Phase 4 MVP with entity-level validation
# ========================================

CONFIG = {
    # Model configuration
    'model_name_or_path': 'roberta-base',
    'n_metadata_features': 28,  # CRITICAL: Must be 28 (usable features)
    'num_classes': 2,  # Binary classification
    'num_ner_labels': 3,  # BIO tagging: O, B-COM, I-COM
    'n_boolean_features': 10,  # For auxiliary task
    'n_numerical_features': 2,  # For auxiliary task
    
    # Training hyperparameters
    'learning_rate': 2e-5,  # Aligned with V2 baseline
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'batch_size': 32,  # A100 optimization (use 16 for T4/V100)
    'epochs': 3 if TEST_MODE else 30,
    'gradient_clipping': 1.0,
    
    # Loss weighting (fixed for Phase 4 MVP)
    'lambda_classification': 0.3,  # Classification task weight
    'lambda_ner': 0.7,  # NER task weight (higher priority)
    'lambda_auxiliary': 0.1,  # Auxiliary regularization weight
    
    # Task-specific dropout
    'classification_dropout': 0.3,
    'ner_dropout': 0.1,
    
    # Early stopping
    'patience': 10,
    'min_delta': 0.001,
    
    # Data configuration
    'max_length_classif': 256,
    'max_length_ner': 512,
    'oversample_ner': True,  # Balance NER/classification samples
    
    # NEW: Entity-level validation configuration
    'use_entity_level_validation': True,  # Use entity-level F1 (not token-level)
    'use_crf': True,  # Add CRF layer to NER head
    
    # NEW: Class weights for I-tag boost
    'use_class_weights': True,  # Apply class weights to NER loss
    'i_tag_boost': 2.0,  # Multiplier for I-tag weight to prevent fragmentation
    
    # NEW: B/I ratio monitoring
    'monitor_bi_ratio': True,  # Track B-tag to I-tag ratio
    'target_bi_ratio': 0.5,  # Target ratio: 1:2 (1 B for every 2 I tags)
    
    # Mode flags
    'test_mode': TEST_MODE,
    'use_mixed_precision': True,  # A100 optimization
    'session_id': SESSION_ID
}

print("="*80)
print("PHASE 4: MULTI-TASK LEARNING CONFIGURATION (FIXED)")
print("="*80)
print(f"\nMode: {'TEST (50 samples, 3 epochs)' if TEST_MODE else 'FULL TRAINING (1,634 samples, 30 epochs)'}")
print(f"\nModel: {CONFIG['model_name_or_path']}")
print(f"Metadata features: {CONFIG['n_metadata_features']}")
print(f"\nTraining:")
print(f"  - Learning rate: {CONFIG['learning_rate']}")
print(f"  - Batch size: {CONFIG['batch_size']}")
print(f"  - Epochs: {CONFIG['epochs']}")
print(f"  - Warmup steps: {CONFIG['warmup_steps']}")
print(f"  - Gradient clipping: {CONFIG['gradient_clipping']}")
print(f"\nLoss Weighting:")
print(f"  - Classification (λ₁): {CONFIG['lambda_classification']}")
print(f"  - NER (λ₂): {CONFIG['lambda_ner']}")
print(f"  - Auxiliary (λ₃): {CONFIG['lambda_auxiliary']}")
print(f"\nDropout:")
print(f"  - Classification: {CONFIG['classification_dropout']}")
print(f"  - NER: {CONFIG['ner_dropout']}")
print(f"\n🆕 NEW: Entity-Level Validation:")
print(f"  - CRF layer: {CONFIG['use_crf']} (learns valid BIO transitions)")
print(f"  - Entity-level F1: {CONFIG['use_entity_level_validation']} (counts full entities)")
print(f"  - Class weights: {CONFIG['use_class_weights']} (I-tag boost: {CONFIG['i_tag_boost']}x)")
print(f"  - B/I ratio monitoring: {CONFIG['monitor_bi_ratio']} (target: {CONFIG['target_bi_ratio']})")
print(f"\nOptimizations:")
print(f"  - Mixed precision: {CONFIG['use_mixed_precision']} (A100)")
print(f"  - NER oversampling: {CONFIG['oversample_ner']}")

# V2 Baseline for comparison
V2_BASELINE = {
    'classification_f1': 0.898,
    'ner_entity_f1': 0.1677  # Original token-level was 0.749, entity-level was 16.77%
}

print(f"\n{'='*80}")
print("V2 BASELINE TARGETS")
print(f"{'='*80}")
print(f"Classification F1: {V2_BASELINE['classification_f1']:.3f} (maintain)")
print(f"NER Entity F1:    {V2_BASELINE['ner_entity_f1']:.3f} (target ≥ 0.600)")
print(f"\nPhase 4 Success Criteria:")
print(f"  ✓ Classification F1 ≥ 0.890 (within 1% of V2)")
print(f"  ✓ NER Entity F1 ≥ 0.600 (significant improvement from 16.77%)")
print(f"  ✓ Multi-task learning enabled (shared encoder)")
print(f"  ✓ Metadata integration (28 features)")
print(f"  ✓ CRF layer prevents entity fragmentation")

# Estimate training time
if TEST_MODE:
    estimated_time = "5-10 minutes"
else:
    estimated_time = "2-3 hours (A100) or 4-6 hours (T4/V100)"

print(f"\nEstimated training time: {estimated_time}")
print(f"{'='*80}")

## Cell 3: Environment Setup and GPU Optimization

In [ ]:
import sys
import subprocess
import torch

# Add src to path
if '/content/drive/MyDrive/inventory_2022/src' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/inventory_2022/src')

print("📦 Installing/upgrading dependencies...")
# Install required packages (pin transformers for compatibility)
subprocess.run([
    'pip', 'install', '-q',
    'transformers==4.35.2',
    'datasets',
    'evaluate',
    'seqeval',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'pandas',
    'tqdm',
    'pytorch-crf'  # NEW: CRF layer for BIO tag transition learning
], check=False)

print("\n🔧 Importing modules...")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import logging
from datetime import datetime
from tqdm import tqdm

# Import Phase 4 modules with error handling
try:
    from src.models.multitask_model import BiomedicalMultiTaskModel, create_model
    from src.data.multitask_dataloader import create_multitask_dataloaders
    from src.train_multitask import MultiTaskTrainer
    from src.evaluate_multitask import MultiTaskEvaluator, generate_evaluation_report
    print("✅ Phase 4 modules imported successfully")
except ImportError as e:
    print(f"❌ Failed to import Phase 4 modules: {e}")
    print("\n🔍 Checking available modules:")
    import os
    src_path = '/content/drive/MyDrive/inventory_2022/src'
    if os.path.exists(src_path):
        print(f"   Available files in {src_path}:")
        for item in os.listdir(src_path):
            print(f"     - {item}")
    else:
        print(f"   ❌ src directory not found at {src_path}")
    raise

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# GPU Setup
print("\n" + "="*60)
print("GPU CONFIGURATION")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    # Configure mixed precision based on GPU type
    is_a100 = 'A100' in gpu_name
    if is_a100:
        print(f"\n   🚀 A100 GPU DETECTED - Optimizations enabled:")
        print(f"      ✅ Mixed precision training (FP16)")
        print(f"      ✅ Batch size: {CONFIG['batch_size']}")
        print(f"      ✅ Expected training time: ~1-1.5 hours (30 epochs)")
        CONFIG['use_mixed_precision'] = True
    else:
        print(f"\n   💡 Non-A100 GPU detected: {gpu_name}")
        # Conservative settings for T4/V100
        if 'T4' in gpu_name or 'V100' in gpu_name:
            CONFIG['use_mixed_precision'] = True  # Usually safe
            print(f"      ✅ Mixed precision: enabled")
            print(f"      ⚠️  Monitor for NaN losses (uncommon but possible)")
            if CONFIG['batch_size'] > 16:
                print(f"      ⚠️  Batch size {CONFIG['batch_size']} may cause OOM")
                print(f"         Consider reducing to 16 if training fails")
            print(f"      ⏱  Expected training time: ~3-5 hours (30 epochs)")
        else:
            CONFIG['use_mixed_precision'] = False  # Safer for unknown GPUs
            print(f"      ⚠️  Mixed precision: DISABLED (unknown GPU)")
            print(f"      💡 Training will be slower but more stable")
            print(f"      ⏱  Expected training time: ~6-8 hours (30 epochs)")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    print(f"\n   Memory after cleanup:")
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"      Allocated: {allocated:.2f} GB")
    print(f"      Reserved:  {reserved:.2f} GB")
    print(f"      Free:      {gpu_memory - reserved:.2f} GB")
    
    device = torch.device('cuda')
else:
    print("⚠️ No GPU available - training will be slow")
    print("   Consider using a GPU runtime for Phase 4 training")
    device = torch.device('cpu')
    CONFIG['use_mixed_precision'] = False  # Disable on CPU

CONFIG['device'] = device
print(f"\n✅ Environment setup complete")
print(f"   Device: {device}")

## Cell 4: Data Loading and 80/20 Split Creation

In [ ]:
print("="*80)
print("DATA LOADING AND PREPARATION")
print("="*80)

# Data paths
CLASSIF_TRAIN_PATH = "data/augmented/classif_train_with_metadata.csv"
NER_TRAIN_PATH = "data/augmented/ner_train_with_metadata.csv"

print(f"\n📂 Data Sources:")
print(f"   Classification: {CLASSIF_TRAIN_PATH}")
print(f"   NER: {NER_TRAIN_PATH}")

# Verify files exist with helpful instructions
if not Path(CLASSIF_TRAIN_PATH).exists():
    raise FileNotFoundError(
        f"\n❌ Classification data not found: {CLASSIF_TRAIN_PATH}\n\n"
        f"   📋 Required Action: Run data augmentation first\n"
        f"   📝 Command: python src/data_augmentation/augment_with_metadata.py\n\n"
        f"   OR ensure data/augmented/ directory exists with metadata files\n"
        f"   (augmented data is not tracked in git)"
    )
if not Path(NER_TRAIN_PATH).exists():
    raise FileNotFoundError(
        f"\n❌ NER data not found: {NER_TRAIN_PATH}\n\n"
        f"   📋 Required Action: Run data augmentation first\n"
        f"   📝 Command: python src/data_augmentation/augment_with_metadata.py\n\n"
        f"   OR ensure data/augmented/ directory exists with metadata files\n"
        f"   (augmented data is not tracked in git)"
    )

print("\n✅ Data files verified")

# Load full datasets for splitting
print("\n📥 Loading datasets...")
classif_df_full = pd.read_csv(CLASSIF_TRAIN_PATH)
ner_df_full = pd.read_csv(NER_TRAIN_PATH)

print(f"   Classification: {len(classif_df_full)} samples")
print(f"   NER: {len(ner_df_full)} samples")

# Create 80/20 train/val split
from sklearn.model_selection import train_test_split

print("\n✂️  Creating 80/20 train/val splits...")

# Classification split (stratified by curation_score)
classif_train, classif_val = train_test_split(
    classif_df_full,
    test_size=0.2,
    random_state=42,
    stratify=classif_df_full['curation_score']
)

# NER split (stratified by has_resource to maintain balance)
# Assuming has_resource column exists, otherwise use random split
if 'has_resource' in ner_df_full.columns:
    ner_train, ner_val = train_test_split(
        ner_df_full,
        test_size=0.2,
        random_state=42,
        stratify=ner_df_full['has_resource']
    )
else:
    ner_train, ner_val = train_test_split(
        ner_df_full,
        test_size=0.2,
        random_state=42
    )

print(f"\n📊 Split Statistics:")
print(f"\nClassification:")
print(f"   Train: {len(classif_train)} samples")
print(f"   Val:   {len(classif_val)} samples")
print(f"   Positive ratio (train): {classif_train['curation_score'].mean():.3f}")
print(f"   Positive ratio (val):   {classif_val['curation_score'].mean():.3f}")

print(f"\nNER:")
print(f"   Train: {len(ner_train)} samples")
print(f"   Val:   {len(ner_val)} samples")

# Save temporary splits for dataloader
split_dir = Path(EXPERIMENT_DIR) / "splits"
split_dir.mkdir(exist_ok=True)

classif_train_path = split_dir / "classif_train.csv"
classif_val_path = split_dir / "classif_val.csv"
ner_train_path = split_dir / "ner_train.csv"
ner_val_path = split_dir / "ner_val.csv"

classif_train.to_csv(classif_train_path, index=False)
classif_val.to_csv(classif_val_path, index=False)
ner_train.to_csv(ner_train_path, index=False)
ner_val.to_csv(ner_val_path, index=False)

print(f"\n💾 Splits saved to: {split_dir}")

# TEST_MODE: Use first 50 samples
if TEST_MODE:
    print(f"\n⚡ TEST_MODE: Using first 50 samples per task")
    classif_train = classif_train.head(50)
    classif_val = classif_val.head(25)
    ner_train = ner_train.head(50)
    ner_val = ner_val.head(25)
    
    # Re-save reduced splits
    classif_train.to_csv(classif_train_path, index=False)
    classif_val.to_csv(classif_val_path, index=False)
    ner_train.to_csv(ner_train_path, index=False)
    ner_val.to_csv(ner_val_path, index=False)
    
    print(f"   Classification: {len(classif_train)} train, {len(classif_val)} val")
    print(f"   NER: {len(ner_train)} train, {len(ner_val)} val")

print("\n✅ Data preparation complete")

## Cell 5: NEW - Compute NER Class Weights for I-Tag Boost

In [ ]:
print("="*80)
print("🆕 NEW: COMPUTING NER CLASS WEIGHTS")
print("="*80)

def compute_ner_class_weights(ner_train_path, boost_i_tag=2.0):
    """
    Compute class weights to boost I-tag predictions.
    
    This prevents entity fragmentation by encouraging the model to continue
    multi-word entities with I-tags instead of starting new entities with B-tags.
    
    Args:
        ner_train_path: Path to NER training data
        boost_i_tag: Multiplier for I-tag weight (default 2.0)
    
    Returns:
        torch.Tensor of class weights [num_labels]
    """
    # Load training data
    ner_df = pd.read_csv(ner_train_path)
    
    # Count tag occurrences
    tag_counts = {}
    for _, row in ner_df.iterrows():
        # Parse NER tags (handle string representation of list)
        tags_str = row['ner_tags']
        if isinstance(tags_str, str):
            import ast
            tags = ast.literal_eval(tags_str)
        else:
            tags = tags_str
            
        for tag in tags:
            tag_counts[tag] = tag_counts.get(tag, 0) + 1
    
    # Compute inverse frequency weights
    total = sum(tag_counts.values())
    weights = {}
    
    # Map tag names to IDs (assumes standard BIO encoding)
    tag_to_id = {'O': 0, 'B-COM': 1, 'I-COM': 2}
    
    for tag_name, tag_id in tag_to_id.items():
        count = tag_counts.get(tag_name, 1)
        weights[tag_id] = total / (3 * count)  # Inverse frequency
    
    # Boost I-tag weight to prevent fragmentation
    weights[2] *= boost_i_tag  # I-COM gets extra boost
    
    # Normalize weights (maintain relative ratios but sum to 3)
    weight_tensor = torch.tensor([weights[0], weights[1], weights[2]], dtype=torch.float32)
    weight_tensor = weight_tensor / weight_tensor.sum() * 3
    
    print(f"📊 NER Class Weights:")
    print(f"   O (tag 0):     {weight_tensor[0]:.3f}")
    print(f"   B-COM (tag 1): {weight_tensor[1]:.3f}")
    print(f"   I-COM (tag 2): {weight_tensor[2]:.3f} (boosted {boost_i_tag}x)")
    print(f"\n   Tag counts:")
    for tag_name, count in tag_counts.items():
        print(f"     {tag_name}: {count:,} ({count/total*100:.1f}%)")
    
    print(f"\n   👉 Higher I-COM weight encourages multi-word entities")
    print(f"   👉 Prevents fragmentation like: ['Mouse'] ['Phenome'] ['Database']")
    
    return weight_tensor

# Compute weights if enabled
if CONFIG['use_class_weights']:
    ner_class_weights = compute_ner_class_weights(
        ner_train_path,
        boost_i_tag=CONFIG['i_tag_boost']
    ).to(device)
    
    print(f"\n✅ Class weights computed and moved to {device}")
else:
    ner_class_weights = None
    print(f"\n⚠️  Class weights disabled (use_class_weights=False)")

# Store in config for trainer
CONFIG['ner_class_weights'] = ner_class_weights

print("\n✅ NER class weights setup complete")

## Cell 6: NEW - Define CRF-Enhanced NER Head

In [ ]:
print("="*80)
print("🆕 NEW: DEFINING CRF-ENHANCED NER HEAD")
print("="*80)

import torch.nn as nn
from torchcrf import CRF

class CRFNERHead(nn.Module):
    """
    Token-level classification head for NER with CRF layer.
    
    The CRF (Conditional Random Field) layer learns valid BIO tag transitions:
    - VALID: O → B-COM, B-COM → I-COM, I-COM → I-COM
    - PENALIZED: B-COM → B-COM (without I-COM), I-COM → B-COM (entity break)
    
    This prevents entity fragmentation by enforcing that multi-word entities
    use continuous I-tags after the initial B-tag.
    
    Architecture:
    - Broadcast metadata to all tokens
    - Linear layer: hidden_size → num_labels (emission scores)
    - CRF layer: learns transition scores between tags
    - Lower dropout (0.1) to preserve token-level information
    
    Args:
        hidden_size: Input dimension (768)
        num_labels: Number of BIO tags (3: O, B-COM, I-COM)
        dropout: Dropout rate (0.1 recommended for NER)
    """
    
    def __init__(self, hidden_size: int, num_labels: int = 3, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)  # Emission scores
        self.crf = CRF(num_labels, batch_first=True)  # CRF for transition scores
        
        print(f"   ✅ CRFNERHead initialized:")
        print(f"      - Hidden size: {hidden_size}")
        print(f"      - Num labels: {num_labels} (O, B-COM, I-COM)")
        print(f"      - Dropout: {dropout}")
        print(f"      - CRF layer: ENABLED (learns BIO transitions)")
    
    def forward(self, sequence_output: torch.Tensor, metadata_embedding: torch.Tensor,
                labels: torch.Tensor = None, attention_mask: torch.Tensor = None):
        """
        Forward pass with CRF loss or Viterbi decoding.
        
        Args:
            sequence_output: [batch_size, seq_len, hidden_size] - All token embeddings
            metadata_embedding: [batch_size, hidden_size] - Projected metadata
            labels: [batch_size, seq_len] - Ground truth BIO tags (for training)
            attention_mask: [batch_size, seq_len] - Mask for padding tokens
        
        Returns:
            If training (labels provided):
                loss: CRF negative log-likelihood
            If inference (labels not provided):
                predictions: [batch_size, seq_len] - Viterbi decoded tags
        """
        # Broadcast metadata to all tokens
        batch_size, seq_len, hidden_size = sequence_output.shape
        metadata_expanded = metadata_embedding.unsqueeze(1).expand(batch_size, seq_len, -1)
        
        # Add metadata information to each token (residual connection)
        enhanced_sequence = sequence_output + metadata_expanded
        
        # Compute emission scores
        x = self.dropout(enhanced_sequence)
        emissions = self.classifier(x)  # [batch_size, seq_len, num_labels]
        
        if labels is not None:
            # Training: Compute CRF loss (negative log-likelihood)
            mask = attention_mask.bool() if attention_mask is not None else None
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            # Inference: Viterbi decoding for best tag sequence
            mask = attention_mask.bool() if attention_mask is not None else None
            predictions = self.crf.decode(emissions, mask=mask)
            
            # Convert list of sequences to tensor (pad if needed)
            max_len = max(len(seq) for seq in predictions)
            predictions_tensor = torch.full((batch_size, max_len), fill_value=0, 
                                           dtype=torch.long, device=emissions.device)
            for i, seq in enumerate(predictions):
                predictions_tensor[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
            
            return predictions_tensor

print("\n✅ CRF-enhanced NER head defined")
print("\n💡 How CRF prevents entity fragmentation:")
print("   1. Emission scores: Token classifier suggests likely tags")
print("   2. Transition scores: CRF learns valid tag sequences")
print("   3. Viterbi decoding: Finds globally optimal tag sequence")
print("   4. Result: Multi-word entities stay together (B-COM → I-COM → I-COM)")
print("\n   Example:")
print("     WITHOUT CRF: [B-COM] [B-COM] [B-COM] = 3 entities (WRONG)")
print("                  Mouse   Phenome  Database")
print("\n     WITH CRF:    [B-COM] [I-COM] [I-COM] = 1 entity (CORRECT)")
print("                  Mouse   Phenome  Database")

## Cell 7: NEW - Define Entity-Level Validation Functions

In [ ]:
print("="*80)
print("🆕 NEW: DEFINING ENTITY-LEVEL VALIDATION FUNCTIONS")
print("="*80)

from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from seqeval.scheme import IOB2

def compute_entity_level_metrics(predictions, labels, id2tag, attention_mask):
    """
    Compute entity-level F1 using seqeval.
    
    Unlike token-level F1, entity-level F1 counts full entities:
    - "Mouse Phenome Database" = 1 entity (not 3 tokens)
    - All tokens must match for the entity to count as correct
    
    Args:
        predictions: Model predictions [batch_size, seq_len] or list of lists
        labels: Ground truth labels [batch_size, seq_len]
        id2tag: Mapping from label IDs to BIO tags
        attention_mask: Attention mask [batch_size, seq_len]
    
    Returns:
        dict with entity_f1, entity_precision, entity_recall
    """
    # Convert predictions and labels to tag sequences
    pred_tags = []
    gold_tags = []
    
    # Handle both tensor and list inputs
    if torch.is_tensor(predictions):
        predictions = predictions.cpu().numpy()
    if torch.is_tensor(labels):
        labels = labels.cpu().numpy()
    if torch.is_tensor(attention_mask):
        attention_mask = attention_mask.cpu().numpy()
    
    for pred_seq, gold_seq, mask in zip(predictions, labels, attention_mask):
        # Get actual length (excluding padding)
        actual_len = int(mask.sum())
        
        # Convert IDs to tags
        pred_seq_tags = [id2tag.get(int(p), 'O') for p in pred_seq[:actual_len]]
        gold_seq_tags = [id2tag.get(int(g), 'O') for g in gold_seq[:actual_len]]
        
        pred_tags.append(pred_seq_tags)
        gold_tags.append(gold_seq_tags)
    
    # Compute entity-level metrics using seqeval
    # mode='strict': All tokens of entity must match
    entity_f1 = f1_score(gold_tags, pred_tags, mode='strict', scheme=IOB2)
    entity_precision = precision_score(gold_tags, pred_tags, mode='strict', scheme=IOB2)
    entity_recall = recall_score(gold_tags, pred_tags, mode='strict', scheme=IOB2)
    
    return {
        'entity_f1': entity_f1,
        'entity_precision': entity_precision,
        'entity_recall': entity_recall
    }

def compute_bi_ratio(predictions, id2tag):
    """
    Compute ratio of B-tags to I-tags.
    
    Healthy ratio for multi-word entities: ~1:2 or 1:3
    - Too high (e.g., 1:0.5): Entity fragmentation (too many B-tags)
    - Too low (e.g., 1:10): Entities too long (rare but possible)
    
    Args:
        predictions: Model predictions [batch_size, seq_len] or list of lists
        id2tag: Mapping from label IDs to BIO tags
    
    Returns:
        dict with b_count, i_count, bi_ratio
    """
    b_count = 0
    i_count = 0
    
    # Handle both tensor and list inputs
    if torch.is_tensor(predictions):
        predictions = predictions.cpu().numpy()
    
    for pred_seq in predictions:
        for pred in pred_seq:
            tag = id2tag.get(int(pred), 'O')
            if tag.startswith('B-'):
                b_count += 1
            elif tag.startswith('I-'):
                i_count += 1
    
    ratio = b_count / max(i_count, 1)  # Avoid division by zero
    
    return {
        'b_count': b_count,
        'i_count': i_count,
        'bi_ratio': ratio
    }

# Define ID to tag mapping (standard BIO encoding)
id2tag = {
    0: 'O',
    1: 'B-COM',
    2: 'I-COM'
}

tag2id = {v: k for k, v in id2tag.items()}

print("\n✅ Entity-level validation functions defined")
print(f"\n📊 ID to Tag Mapping:")
for tag_id, tag_name in id2tag.items():
    print(f"   {tag_id} → {tag_name}")

print("\n💡 Entity-level vs Token-level F1:")
print("   TOKEN-LEVEL (original):")
print("     Prediction: [B-COM] [B-COM] [B-COM]")
print("     Gold:       [B-COM] [I-COM] [I-COM]")
print("     Score: 1/3 = 33.3% (1 correct token out of 3)")
print("\n   ENTITY-LEVEL (fixed):")
print("     Prediction: 'Mouse', 'Phenome', 'Database' = 3 entities")
print("     Gold:       'Mouse Phenome Database' = 1 entity")
print("     Score: 0/1 = 0% (no complete entity matches)")
print("\n   👉 Entity-level F1 is stricter and more meaningful!")

# Store in config
CONFIG['id2tag'] = id2tag
CONFIG['tag2id'] = tag2id

## Cell 8: Create Multi-Task DataLoaders

In [ ]:
print("="*80)
print("CREATING MULTI-TASK DATALOADERS")
print("="*80)

# Create dataloaders
print(f"\n🔄 Loading tokenizer: {CONFIG['model_name_or_path']}")

train_loader, val_loader = create_multitask_dataloaders(
    classif_train_path=str(classif_train_path),
    ner_train_path=str(ner_train_path),
    classif_val_path=str(classif_val_path),
    ner_val_path=str(ner_val_path),
    tokenizer_name=CONFIG['model_name_or_path'],
    batch_size=CONFIG['batch_size'],
    test_mode=TEST_MODE,
    max_length_classif=CONFIG['max_length_classif'],
    max_length_ner=CONFIG['max_length_ner'],
    oversample_ner=CONFIG['oversample_ner'],
    num_workers=0  # Colab compatibility
)

print(f"\n✅ DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Batch size: {CONFIG['batch_size']}")

# Estimate steps per epoch
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * CONFIG['epochs']

print(f"\n📊 Training Schedule:")
print(f"   Steps per epoch: {steps_per_epoch}")
print(f"   Total epochs: {CONFIG['epochs']}")
print(f"   Total steps: {total_steps}")
print(f"   Warmup steps: {CONFIG['warmup_steps']}")
print(f"   Warmup ratio: {CONFIG['warmup_steps']/total_steps:.1%}")

# Test batch
print(f"\n🧪 Testing batch loading...")
test_batch = next(iter(train_loader))
print(f"   Batch task: {test_batch['task'][0]}")
print(f"   Input IDs shape: {test_batch['input_ids'].shape}")
print(f"   Attention mask shape: {test_batch['attention_mask'].shape}")
print(f"   Labels shape: {test_batch['labels'].shape}")
print(f"   Metadata shape: {test_batch['metadata'].shape}")
print(f"   Metadata features: {test_batch['metadata'].shape[1]}")

# Verify metadata feature count matches config
assert test_batch['metadata'].shape[1] == CONFIG['n_metadata_features'], \
    f"Metadata mismatch: expected {CONFIG['n_metadata_features']}, got {test_batch['metadata'].shape[1]}"

print(f"\n✅ Batch loading test passed")

## Cell 9: MODIFIED - Initialize Multi-Task Model with CRF Head

In [ ]:
print("="*80)
print("MODEL INITIALIZATION (WITH CRF HEAD)")
print("="*80)

# Create base model
print(f"\n🏭  Creating BiomedicalMultiTaskModel...")
model = create_model(CONFIG)

# CRITICAL FIX: Replace NER head with CRF-enhanced version
if CONFIG['use_crf']:
    print(f"\n🔧 Replacing NER head with CRF-enhanced version...")
    model.ner_head = CRFNERHead(
        hidden_size=model.hidden_size,
        num_labels=CONFIG['num_ner_labels'],
        dropout=CONFIG['ner_dropout']
    )
    print(f"   ✅ NER head replaced with CRFNERHead")
else:
    print(f"\n⚠️  CRF layer disabled (use_crf=False)")

# Move model to device
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
crf_params = sum(p.numel() for p in model.ner_head.crf.parameters()) if CONFIG['use_crf'] else 0

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
if CONFIG['use_crf']:
    print(f"   CRF parameters: {crf_params:,} (learns tag transitions)")
print(f"   Model size: ~{total_params * 4 / 1024**2:.1f} MB (FP32)")

# Model architecture summary
print(f"\n🏛️  Architecture:")
print(f"   Encoder: {CONFIG['model_name_or_path']}")
print(f"   Hidden size: {model.hidden_size}")
print(f"   Metadata features: {model.n_metadata_features}")
print(f"   Classification classes: {model.num_classes}")
print(f"   NER labels: {model.num_ner_labels}")
print(f"\n   Components:")
print(f"     1. Shared RoBERTa encoder")
print(f"     2. Metadata projection ({CONFIG['n_metadata_features']} → {model.hidden_size})")
print(f"     3. Fusion layer (text + metadata)")
print(f"     4. Classification head (dropout={CONFIG['classification_dropout']})")
if CONFIG['use_crf']:
    print(f"     5. 🆕 CRF-enhanced NER head (dropout={CONFIG['ner_dropout']})")
else:
    print(f"     5. Standard NER head (dropout={CONFIG['ner_dropout']})")
print(f"     6. Auxiliary heads (metadata prediction)")

# Create output directory
output_dir = Path(EXPERIMENT_DIR) / "multitask_training"
output_dir.mkdir(exist_ok=True)

print(f"\n📁 Output directory: {output_dir}")

# Initialize trainer
print(f"\n🎯 Initializing MultiTaskTrainer...")
trainer = MultiTaskTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=CONFIG,
    device=device,
    output_dir=str(output_dir)
)

print(f"\n✅ Trainer initialized:")
print(f"   Optimizer: AdamW (lr={CONFIG['learning_rate']}, wd={CONFIG['weight_decay']})")
print(f"   Scheduler: Linear warmup + decay")
print(f"   Loss weights: λ_classif={CONFIG['lambda_classification']}, λ_ner={CONFIG['lambda_ner']}, λ_aux={CONFIG['lambda_auxiliary']}")
if CONFIG['use_class_weights']:
    print(f"   NER class weights: ENABLED (I-tag boost: {CONFIG['i_tag_boost']}x)")
print(f"   Early stopping: patience={CONFIG['patience']}")
print(f"   Gradient clipping: {CONFIG['gradient_clipping']}")
print(f"   Mixed precision: {CONFIG['use_mixed_precision']}")
if CONFIG['use_entity_level_validation']:
    print(f"   🆕 Entity-level validation: ENABLED")
if CONFIG['monitor_bi_ratio']:
    print(f"   🆕 B/I ratio monitoring: ENABLED (target: {CONFIG['target_bi_ratio']})")

# Validate trainer interface
print(f"\n🔍 Validating trainer interface...")
required_attrs = ['train', 'history', 'optimizer', 'scheduler']
required_methods = ['train']

missing = []
for attr in required_attrs:
    if not hasattr(trainer, attr):
        missing.append(f"attribute '{attr}'")

for method in required_methods:
    if not hasattr(trainer, method) or not callable(getattr(trainer, method)):
        missing.append(f"method '{method}'")

if missing:
    raise AttributeError(
        f"❌ Trainer missing required interfaces: {', '.join(missing)}\n"
        f"   Check that MultiTaskTrainer in src/train_multitask.py is complete"
    )

print(f"   ✅ All required trainer interfaces present")
print(f"   ✅ Ready to begin training")

# Save configuration
config_path = output_dir / "config.json"
with open(config_path, 'w') as f:
    # Convert non-serializable objects to strings
    config_to_save = CONFIG.copy()
    config_to_save['device'] = str(device)
    config_to_save['ner_class_weights'] = str(ner_class_weights) if ner_class_weights is not None else None
    config_to_save['id2tag'] = id2tag
    config_to_save['tag2id'] = tag2id
    json.dump(config_to_save, f, indent=2)

print(f"\n💾 Configuration saved to: {config_path}")
print(f"\n✅ Ready to train with entity-level validation!")

## Cell 10: Training Loop with Entity-Level Validation

**NOTE**: The training loop will need to be modified in `src/train_multitask.py` to:
1. Use CRF loss (already handled by CRFNERHead)
2. Call entity-level validation functions during validation
3. Monitor B/I ratio during training

If you see token-level F1 reported instead of entity-level F1, the trainer needs updating.

In [ ]:
print("="*80)
print(f"STARTING PHASE 4 TRAINING: {SESSION_ID}")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING'}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Device: {device}")
print(f"Validation: {'ENTITY-LEVEL F1' if CONFIG['use_entity_level_validation'] else 'Token-level F1'}")
print(f"CRF Layer: {'ENABLED' if CONFIG['use_crf'] else 'DISABLED'}")
print("="*80)

# Monkey-patch trainer to use entity-level validation if needed
if CONFIG['use_entity_level_validation']:
    # Store original validation method
    original_validate = trainer.validate if hasattr(trainer, 'validate') else None
    
    def entity_level_validate(self):
        """
        Enhanced validation with entity-level F1 and B/I ratio monitoring.
        """
        self.model.eval()
        
        total_loss = 0
        all_ner_predictions = []
        all_ner_labels = []
        all_ner_masks = []
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validation"):
                # Move batch to device
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                metadata = batch['metadata'].to(self.device)
                task = batch['task'][0]
                
                # Forward pass
                outputs = self.model(input_ids, attention_mask, metadata, task)
                
                # Collect NER predictions for entity-level metrics
                if task == 'ner':
                    # Get predictions (CRF head returns decoded sequences)
                    if CONFIG['use_crf']:
                        predictions = self.model.ner_head(
                            outputs['sequence_output'],
                            outputs['metadata_embedding'],
                            labels=None,  # Inference mode
                            attention_mask=attention_mask
                        )
                    else:
                        logits = outputs['logits']
                        predictions = torch.argmax(logits, dim=-1)
                    
                    all_ner_predictions.append(predictions)
                    all_ner_labels.append(labels)
                    all_ner_masks.append(attention_mask)
        
        # Compute entity-level metrics
        if len(all_ner_predictions) > 0:
            # Concatenate all batches
            all_predictions = torch.cat(all_ner_predictions, dim=0)
            all_labels = torch.cat(all_ner_labels, dim=0)
            all_masks = torch.cat(all_ner_masks, dim=0)
            
            # Compute entity-level F1
            entity_metrics = compute_entity_level_metrics(
                all_predictions,
                all_labels,
                CONFIG['id2tag'],
                all_masks
            )
            
            print(f"\n🆕 Entity-Level Metrics:")
            print(f"   Entity F1:        {entity_metrics['entity_f1']:.4f}")
            print(f"   Entity Precision: {entity_metrics['entity_precision']:.4f}")
            print(f"   Entity Recall:    {entity_metrics['entity_recall']:.4f}")
            
            # Compute B/I ratio
            if CONFIG['monitor_bi_ratio']:
                bi_stats = compute_bi_ratio(all_predictions, CONFIG['id2tag'])
                print(f"\n📊 B/I Ratio Monitoring:")
                print(f"   B-tags: {bi_stats['b_count']}")
                print(f"   I-tags: {bi_stats['i_count']}")
                print(f"   B/I Ratio: {bi_stats['bi_ratio']:.3f} (target: {CONFIG['target_bi_ratio']})")
                
                if bi_stats['bi_ratio'] > 1.0:
                    print(f"   ⚠️  High B/I ratio indicates entity fragmentation!")
                elif bi_stats['bi_ratio'] < 0.3:
                    print(f"   ⚠️  Low B/I ratio indicates very long entities")
                else:
                    print(f"   ✅ B/I ratio looks healthy")
            
            return entity_metrics['entity_f1']
        
        return 0.0
    
    # Replace validation method
    if hasattr(trainer, 'validate'):
        import types
        trainer.validate = types.MethodType(entity_level_validate, trainer)
        print("\n✅ Trainer validation method replaced with entity-level version")

# Start training
import time
start_time = time.time()

try:
    trainer.train(num_epochs=CONFIG['epochs'])
    
    training_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total time: {training_time/60:.1f} minutes ({training_time/3600:.2f} hours)")
    print(f"\n📊 Best Results:")
    print(f"   Classification F1: {trainer.best_classif_f1:.4f}")
    print(f"   NER Entity F1:     {trainer.best_ner_f1:.4f} (entity-level)")
    print(f"   Combined F1:       {trainer.best_combined_f1:.4f}")
    
    # Compare to V2 baseline
    print(f"\n📈 Comparison to V2 Baseline:")
    classif_delta = trainer.best_classif_f1 - V2_BASELINE['classification_f1']
    classif_pct = (classif_delta / V2_BASELINE['classification_f1']) * 100
    ner_delta = trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1']
    ner_pct = (ner_delta / V2_BASELINE['ner_entity_f1']) * 100
    
    print(f"   Classification: {trainer.best_classif_f1:.4f} vs {V2_BASELINE['classification_f1']:.4f} ({classif_delta:+.4f}, {classif_pct:+.2f}%)")
    print(f"   NER Entity:     {trainer.best_ner_f1:.4f} vs {V2_BASELINE['ner_entity_f1']:.4f} ({ner_delta:+.4f}, {ner_pct:+.2f}%)")
    
    # Phase 4 success criteria
    print(f"\n🎯 Phase 4 Success Criteria:")
    classif_pass = trainer.best_classif_f1 >= 0.890
    ner_pass = trainer.best_ner_f1 >= 0.600  # Updated target: 60% entity-level F1
    print(f"   Classification F1 ≥ 0.890: {'✅ PASS' if classif_pass else '❌ FAIL'} ({trainer.best_classif_f1:.4f})")
    print(f"   NER Entity F1 ≥ 0.600:     {'✅ PASS' if ner_pass else '❌ FAIL'} ({trainer.best_ner_f1:.4f})")
    print(f"   Multi-task learning:         ✅ ENABLED")
    print(f"   Metadata integration:        ✅ ENABLED (28 features)")
    print(f"   CRF layer:                   {'✅ ENABLED' if CONFIG['use_crf'] else '❌ DISABLED'}")
    print(f"   Entity-level validation:     {'✅ ENABLED' if CONFIG['use_entity_level_validation'] else '❌ DISABLED'}")
    
    overall_pass = classif_pass and ner_pass
    print(f"\n   Overall Phase 4 Status: {'✅ SUCCESS' if overall_pass else '⚠️  PARTIAL SUCCESS'}")
    
except torch.cuda.OutOfMemoryError:
    print("\n" + "="*60)
    print("❌ GPU OUT OF MEMORY!")
    print("="*60)
    print("Current configuration:")
    print(f"   Batch size: {CONFIG['batch_size']}")
    print(f"   Max length (classification): {CONFIG['max_length_classif']}")
    print(f"   Max length (NER): {CONFIG['max_length_ner']}")
    print("\n💡 Suggestions to fix:")
    print("   1. Reduce batch_size from 32 to 16 (edit Cell 2)")
    print("   2. Reduce max_length_ner from 512 to 384 (edit Cell 2)")
    print("   3. Disable mixed_precision (set to False in Cell 3)")
    print("   4. Restart runtime: Runtime → Restart runtime")
    print("\n📊 Current GPU usage:")
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"   Allocated: {allocated:.2f} GB")
        print(f"   Reserved: {reserved:.2f} GB")
    raise

except RuntimeError as e:
    error_str = str(e)
    if "CUDA" in error_str or "cuda" in error_str:
        print("\n" + "="*60)
        print(f"❌ CUDA ERROR")
        print("="*60)
        print(f"Error message: {error_str}")
        print("\n💡 This may indicate:")
        print("   - GPU disconnection")
        print("   - Driver crash")
        print("   - Memory corruption")
        print("   - Incompatible CUDA operations")
        print("\n🔧 Try these fixes:")
        print("   1. Runtime → Restart runtime")
        print("   2. Disable mixed precision (Cell 3)")
        print("   3. Check GPU availability (Runtime → View resources)")
    raise
    
except KeyboardInterrupt:
    print("\n" + "="*60)
    print("⚠️  TRAINING INTERRUPTED BY USER")
    print("="*60)
    training_time = time.time() - start_time
    print(f"Time elapsed: {training_time/60:.1f} minutes")
    
    # Save interrupt checkpoint
    print(f"\n💾 Saving interrupt checkpoint...")
    interrupt_path = output_dir / "checkpoint_interrupt.pt"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'epoch': len(trainer.history.get('train_loss', [])),
        'config': CONFIG,
        'training_time': training_time
    }, interrupt_path)
    print(f"✅ Checkpoint saved to: {interrupt_path}")
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ TRAINING FAILED")
    print("="*60)
    print(f"Error: {str(e)}")
    print("\nFull traceback:")
    import traceback
    traceback.print_exc()
    raise

## Cell 11: Evaluation with Entity-Level Metrics

In [ ]:
print("="*80)
print("FINAL EVALUATION (ENTITY-LEVEL)")
print("="*80)

# Load best combined checkpoint
best_checkpoint_path = output_dir / "checkpoint_best_combined.pt"

if best_checkpoint_path.exists():
    print(f"\n📥 Loading best checkpoint: {best_checkpoint_path}")
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"   Checkpoint epoch: {checkpoint['epoch']}")
    print(f"   Checkpoint metrics:")
    for key, value in checkpoint['metrics'].items():
        print(f"     {key}: {value:.4f}")
else:
    print(f"\n⚠️  Best checkpoint not found, using current model state")

# Evaluate on validation set with entity-level metrics
print(f"\n🔍 Evaluating on validation set...")
model.eval()

all_ner_predictions = []
all_ner_labels = []
all_ner_masks = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluation"):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        metadata = batch['metadata'].to(device)
        task = batch['task'][0]
        
        if task == 'ner':
            # Forward pass
            outputs = model(input_ids, attention_mask, metadata, task)
            
            # Get predictions
            if CONFIG['use_crf']:
                predictions = model.ner_head(
                    outputs['sequence_output'],
                    outputs['metadata_embedding'],
                    labels=None,
                    attention_mask=attention_mask
                )
            else:
                logits = outputs['logits']
                predictions = torch.argmax(logits, dim=-1)
            
            all_ner_predictions.append(predictions)
            all_ner_labels.append(labels)
            all_ner_masks.append(attention_mask)

# Compute final entity-level metrics
if len(all_ner_predictions) > 0:
    all_predictions = torch.cat(all_ner_predictions, dim=0)
    all_labels = torch.cat(all_ner_labels, dim=0)
    all_masks = torch.cat(all_ner_masks, dim=0)
    
    # Entity-level F1
    entity_metrics = compute_entity_level_metrics(
        all_predictions,
        all_labels,
        CONFIG['id2tag'],
        all_masks
    )
    
    # B/I ratio
    bi_stats = compute_bi_ratio(all_predictions, CONFIG['id2tag'])
    
    print(f"\n{'='*80}")
    print("FINAL ENTITY-LEVEL RESULTS")
    print(f"{'='*80}")
    print(f"\n🆕 Entity-Level Metrics:")
    print(f"   Entity F1:        {entity_metrics['entity_f1']:.4f}")
    print(f"   Entity Precision: {entity_metrics['entity_precision']:.4f}")
    print(f"   Entity Recall:    {entity_metrics['entity_recall']:.4f}")
    
    print(f"\n📊 B/I Ratio Analysis:")
    print(f"   B-tags: {bi_stats['b_count']:,}")
    print(f"   I-tags: {bi_stats['i_count']:,}")
    print(f"   B/I Ratio: {bi_stats['bi_ratio']:.3f}")
    print(f"   Target: {CONFIG['target_bi_ratio']}")
    
    if bi_stats['bi_ratio'] > 1.0:
        print(f"   ⚠️  Entity fragmentation detected (too many B-tags)")
    elif bi_stats['bi_ratio'] < 0.3:
        print(f"   ⚠️  Very long entities (too few B-tags)")
    else:
        print(f"   ✅ Healthy B/I ratio for multi-word entities")
    
    # Compare to baseline
    print(f"\n📈 Comparison to V2 Baseline:")
    improvement = entity_metrics['entity_f1'] - V2_BASELINE['ner_entity_f1']
    improvement_pct = (improvement / V2_BASELINE['ner_entity_f1']) * 100
    print(f"   Baseline Entity F1: {V2_BASELINE['ner_entity_f1']:.4f}")
    print(f"   Phase 4 Entity F1:  {entity_metrics['entity_f1']:.4f}")
    print(f"   Improvement:        {improvement:+.4f} ({improvement_pct:+.1f}%)")
    
    if entity_metrics['entity_f1'] >= 0.600:
        print(f"\n   ✅ SUCCESS: Entity F1 ≥ 0.600 target achieved!")
    else:
        print(f"\n   ⚠️  Below target: Entity F1 < 0.600")
        print(f"      Consider: Increase I-tag boost, adjust CRF learning rate")
    
    # Save results
    results = {
        'entity_f1': float(entity_metrics['entity_f1']),
        'entity_precision': float(entity_metrics['entity_precision']),
        'entity_recall': float(entity_metrics['entity_recall']),
        'b_count': int(bi_stats['b_count']),
        'i_count': int(bi_stats['i_count']),
        'bi_ratio': float(bi_stats['bi_ratio']),
        'baseline_entity_f1': float(V2_BASELINE['ner_entity_f1']),
        'improvement': float(improvement),
        'improvement_pct': float(improvement_pct)
    }
    
    results_path = output_dir / "entity_level_results.json"
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n💾 Results saved to: {results_path}")

print("\n✅ Evaluation complete")

## Cell 12: Visualization - Training Curves with Entity-Level Metrics

In [ ]:
# This cell is identical to Cell 9 in original notebook
# Just update the title and labels to reflect entity-level validation

print("="*80)
print("TRAINING VISUALIZATION")
print("="*80)

# Load training history
history = trainer.history

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'Phase 4 Multi-Task Training (Entity-Level) - {SESSION_ID}', fontsize=16, fontweight='bold')

# Plot 1: Total Loss
ax = axes[0, 0]
epochs = range(1, len(history['train_loss']) + 1)
ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Total Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Task-Specific Losses
ax = axes[0, 1]
ax.plot(epochs, history['train_classif_loss'], 'g-', linewidth=2, label='Classification Loss')
ax.plot(epochs, history['train_ner_loss'], 'r-', linewidth=2, label='NER Loss (CRF)')
ax.plot(epochs, history['train_aux_loss'], 'orange', linewidth=2, label='Auxiliary Loss', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Task-Specific Losses')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Validation F1 Scores (Entity-Level)
ax = axes[1, 0]
if history.get('val_classif_f1'):
    ax.plot(epochs, history['val_classif_f1'], 'g-', linewidth=2, marker='o', label='Classification F1')
    ax.axhline(y=V2_BASELINE['classification_f1'], color='g', linestyle='--', alpha=0.5, label='V2 Classif Baseline')
if history.get('val_ner_f1'):
    ax.plot(epochs, history['val_ner_f1'], 'r-', linewidth=2, marker='s', label='NER Entity F1')
    ax.axhline(y=V2_BASELINE['ner_entity_f1'], color='r', linestyle='--', alpha=0.5, label='V2 NER Baseline')
    ax.axhline(y=0.600, color='orange', linestyle=':', alpha=0.5, label='Target (60%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.set_title('Validation F1 Scores (Entity-Level)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.0, 1.0])

# Plot 4: Summary Statistics
ax = axes[1, 1]
ax.axis('off')

summary_text = f"""
PHASE 4 RESULTS (ENTITY-LEVEL)
{'='*35}

Classification F1: {trainer.best_classif_f1:.4f}
  V2 Baseline:     {V2_BASELINE['classification_f1']:.4f}
  Delta:           {trainer.best_classif_f1 - V2_BASELINE['classification_f1']:+.4f}

NER Entity F1:    {trainer.best_ner_f1:.4f}
  V2 Baseline:     {V2_BASELINE['ner_entity_f1']:.4f}
  Delta:           {trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1']:+.4f}
  Improvement:     {(trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1'])/V2_BASELINE['ner_entity_f1']*100:+.1f}%

Combined F1:      {trainer.best_combined_f1:.4f}

{'='*35}
CONFIGURATION
{'='*35}

CRF Layer:          ✅ {CONFIG['use_crf']}
Entity-Level Val:   ✅ {CONFIG['use_entity_level_validation']}
Class Weights:      ✅ {CONFIG['use_class_weights']}
I-tag Boost:        {CONFIG['i_tag_boost']}x

Loss Weighting:
  Classification (λ₁): {CONFIG['lambda_classification']}
  NER (λ₂):           {CONFIG['lambda_ner']}
  Auxiliary (λ₃):     {CONFIG['lambda_auxiliary']}

Training:
  Epochs:             {CONFIG['epochs']}
  Batch size:         {CONFIG['batch_size']}
  Learning rate:      {CONFIG['learning_rate']}

Phase 4 Success:
  NER Entity F1 ≥ 60%: {'✅' if trainer.best_ner_f1 >= 0.600 else '❌'}
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()

# Save figure
curves_path = output_dir / "training_curves_entity_level.png"
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Training curves saved to: {curves_path}")

plt.show()

print("\n✅ Visualization complete")

## Cell 13: Archive Session to Google Drive

In [ ]:
# This cell is nearly identical to Cell 10 in original notebook
# Just update the summary to reflect entity-level validation

import shutil

print("="*80)
print("SESSION ARCHIVAL")
print("="*80)

print(f"\n📦 Archiving session to Google Drive...")
print(f"   Source: {output_dir}")
print(f"   Destination: {ARCHIVE_DIR}")

# Copy entire output directory to archive
archive_output = Path(ARCHIVE_DIR) / "multitask_training"
if archive_output.exists():
    shutil.rmtree(archive_output)

shutil.copytree(output_dir, archive_output)

print(f"\n✅ Archived training outputs")

# Copy splits for reproducibility
archive_splits = Path(ARCHIVE_DIR) / "splits"
if archive_splits.exists():
    shutil.rmtree(archive_splits)

shutil.copytree(split_dir, archive_splits)

print(f"✅ Archived data splits")

# Create comprehensive session summary
summary_path = Path(ARCHIVE_DIR) / "SESSION_SUMMARY.md"

summary_content = f"""
# Phase 4 Multi-Task Training Session (FIXED - Entity-Level Validation)

**Session ID**: {SESSION_ID}  
**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Mode**: {'TEST_MODE' if TEST_MODE else 'FULL_TRAINING'}  
**Device**: {device}  

## Key Improvements

- ✅ **CRF Layer**: Learns valid BIO tag transitions to prevent fragmentation
- ✅ **Entity-Level F1**: Counts full entities (not tokens) for accurate validation
- ✅ **Class Weights**: {CONFIG['i_tag_boost']}x boost for I-tags to encourage multi-word entities
- ✅ **B/I Ratio Monitoring**: Track entity fragmentation during training

## Configuration

### Model
- Base model: {CONFIG['model_name_or_path']}
- Metadata features: {CONFIG['n_metadata_features']}
- Classification classes: {CONFIG['num_classes']}
- NER labels: {CONFIG['num_ner_labels']}
- **CRF layer**: {CONFIG['use_crf']}

### Training
- Learning rate: {CONFIG['learning_rate']}
- Batch size: {CONFIG['batch_size']}
- Epochs: {CONFIG['epochs']}
- Warmup steps: {CONFIG['warmup_steps']}
- Gradient clipping: {CONFIG['gradient_clipping']}
- Mixed precision: {CONFIG['use_mixed_precision']}

### Loss Weighting
- Classification (λ₁): {CONFIG['lambda_classification']}
- NER (λ₂): {CONFIG['lambda_ner']}
- Auxiliary (λ₃): {CONFIG['lambda_auxiliary']}

### Dropout
- Classification: {CONFIG['classification_dropout']}
- NER: {CONFIG['ner_dropout']}

### Entity-Level Validation
- Entity-level F1: {CONFIG['use_entity_level_validation']}
- Class weights: {CONFIG['use_class_weights']}
- I-tag boost: {CONFIG['i_tag_boost']}x
- B/I ratio monitoring: {CONFIG['monitor_bi_ratio']}
- Target B/I ratio: {CONFIG['target_bi_ratio']}

## Results

### Best Metrics
- **Classification F1**: {trainer.best_classif_f1:.4f}
- **NER Entity F1**: {trainer.best_ner_f1:.4f} (entity-level)
- **Combined F1**: {trainer.best_combined_f1:.4f}

### Comparison to V2 Baseline

| Task | Phase 4 | V2 Baseline | Delta | Change |
|------|---------|-------------|-------|--------|
| Classification F1 | {trainer.best_classif_f1:.4f} | {V2_BASELINE['classification_f1']:.4f} | {trainer.best_classif_f1 - V2_BASELINE['classification_f1']:+.4f} | {(trainer.best_classif_f1 - V2_BASELINE['classification_f1'])/V2_BASELINE['classification_f1']*100:+.2f}% |
| NER Entity F1 | {trainer.best_ner_f1:.4f} | {V2_BASELINE['ner_entity_f1']:.4f} | {trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1']:+.4f} | {(trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1'])/V2_BASELINE['ner_entity_f1']*100:+.2f}% |

### Phase 4 Success Criteria

- ✅ Multi-task learning enabled (shared encoder)
- ✅ Metadata integration (28 features)
- ✅ CRF layer prevents entity fragmentation
- ✅ Entity-level F1 validation
- {'✅' if trainer.best_classif_f1 >= 0.890 else '❌'} Classification F1 ≥ 0.890: {trainer.best_classif_f1:.4f}
- {'✅' if trainer.best_ner_f1 >= 0.600 else '❌'} NER Entity F1 ≥ 0.600: {trainer.best_ner_f1:.4f}

**Overall Status**: {'✅ SUCCESS' if (trainer.best_classif_f1 >= 0.890 and trainer.best_ner_f1 >= 0.600) else '⚠️ PARTIAL SUCCESS'}

## Data

### Training Set
- Classification: {len(classif_train)} samples
- NER: {len(ner_train)} samples

### Validation Set
- Classification: {len(classif_val)} samples
- NER: {len(ner_val)} samples

## Files

### Checkpoints
- `checkpoint_best_classification.pt` - Best classification F1 model
- `checkpoint_best_ner.pt` - Best NER Entity F1 model
- `checkpoint_best_combined.pt` - Best combined F1 model
- `checkpoint_final.pt` - Final epoch model

### Outputs
- `training_history.json` - Per-epoch metrics
- `entity_level_results.json` - Final entity-level evaluation metrics
- `training_curves_entity_level.png` - Visualization
- `config.json` - Complete configuration

### Data Splits
- `splits/classif_train.csv` - Classification training data
- `splits/classif_val.csv` - Classification validation data
- `splits/ner_train.csv` - NER training data
- `splits/ner_val.csv` - NER validation data

## Problem Fixed

**Original Issue**: Token-level loss caused entity fragmentation
- Example: "Mouse Phenome Database" → ["Mouse"], ["Phenome"], ["Database"] (3 entities)
- Original entity F1: {V2_BASELINE['ner_entity_f1']:.1%}

**Solution**: CRF + entity-level F1 + class weights
- CRF learns: B-COM → I-COM is valid, B-COM → B-COM is penalized
- Entity-level F1 counts: "Mouse Phenome Database" = 1 entity (not 3 tokens)
- Class weights encourage: B-COM → I-COM → I-COM sequences
- Fixed entity F1: {trainer.best_ner_f1:.1%} ({(trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1'])/V2_BASELINE['ner_entity_f1']*100:+.1f}% improvement)

## Next Steps

1. Review entity_level_results.json for detailed metrics
2. Compare training_curves_entity_level.png to identify convergence
3. Analyze B/I ratio trends (target: ~0.5 for multi-word entities)
4. If Entity F1 ≥ 0.600: Proceed to Phase 5 (inference pipeline)
5. If Entity F1 < 0.600: Adjust I-tag boost or CRF learning rate
6. Document findings and update experiment log

## Notes

- Phase 4 implements multi-task learning with entity-level validation
- CRF layer prevents entity fragmentation by learning valid tag transitions
- Entity-level F1 is stricter but more meaningful than token-level F1
- B/I ratio monitoring helps detect entity fragmentation issues
- All checkpoints and results archived to Google Drive for reproducibility
"""

with open(summary_path, 'w') as f:
    f.write(summary_content)

print(f"\n✅ Session summary created: {summary_path}")

# Display archive contents
print(f"\n📂 Archive contents:")
for item in sorted(Path(ARCHIVE_DIR).rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024*1024)
        rel_path = item.relative_to(ARCHIVE_DIR)
        if size_mb > 0.1:  # Only show files > 100KB
            print(f"   {rel_path} ({size_mb:.1f} MB)")

# Calculate total archive size
total_size = sum(f.stat().st_size for f in Path(ARCHIVE_DIR).rglob('*') if f.is_file())
total_size_mb = total_size / (1024*1024)

print(f"\n{'='*80}")
print("SESSION SUMMARY")
print(f"{'='*80}")
print(f"Session ID: {SESSION_ID}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING'}")
print(f"Training Time: {training_time/60:.1f} minutes")
print(f"Archive Size: {total_size_mb:.1f} MB")
print(f"Archive Location: {ARCHIVE_DIR}")

print(f"\n📊 Final Results:")
print(f"   Classification F1: {trainer.best_classif_f1:.4f} (V2: {V2_BASELINE['classification_f1']:.4f})")
print(f"   NER Entity F1:     {trainer.best_ner_f1:.4f} (V2: {V2_BASELINE['ner_entity_f1']:.4f})")
print(f"   Combined F1:       {trainer.best_combined_f1:.4f}")
print(f"   Entity Improvement: {(trainer.best_ner_f1 - V2_BASELINE['ner_entity_f1'])/V2_BASELINE['ner_entity_f1']*100:+.1f}%")

print(f"\n🎉 PHASE 4 TRAINING SESSION COMPLETE (ENTITY-LEVEL VALIDATION)!")
print(f"\n📝 Next steps:")
print(f"   1. Review {summary_path}")
print(f"   2. Examine entity_level_results.json for detailed metrics")
print(f"   3. Analyze training_curves_entity_level.png for convergence")
print(f"   4. Check B/I ratio trends (should be around {CONFIG['target_bi_ratio']})")
print(f"   5. {('Proceed to Phase 5 inference pipeline' if trainer.best_ner_f1 >= 0.600 else 'Iterate: Adjust I-tag boost or CRF parameters')}")
print(f"   6. Document findings in experiment log")